# Run Model — Core Diagnostics

Self-contained standard diagnostics. The workflow cell runs the configured simulation; use the same inputs as `run_model.ipynb` when reproducing a run.

In [ ]:
# Setup
%load_ext autoreload
%autoreload 2

from src.notebook_config import (
    BALANCE_SHEET_COLUMNS,
    FIGURE_SIZES,
    FISCAL_COLUMNS,
    LABOUR_COLUMNS,
    MACRO_COLUMNS,
    POLICY_COLUMNS,
    SCENARIO_PRESETS,
)
from src.notebook_state import run_notebook_workflow, validate_notebook_state
from src.notebook_workflow import (
    NotebookRunConfig,
    build_permanent_income_forecast_contribution_table,
    build_permanent_income_log_ratio_decomposition_df,
    plot_permanent_income_log_ratio_decomposition,
)
from src.visual_helpers import (
    firm_sector_groups_table,
    plot_agent_timeseries,
    plot_cumulative_insolvent_firms_by_sector,
    plot_firm_credit_to_equity_and_capital,
    plot_output,
    plot_sector_tfp_investment_desired_mb_mc_ratio,
)


## Inputs

In [ ]:
RUN_BENCHMARK = True
RUN_MONTE_CARLO = False
SCENARIO_NAME = "calibrated_consumption"

run_config = NotebookRunConfig(
    seed=232,
    t_max=150,
    country_iso3="FRA",
    run_benchmark=RUN_BENCHMARK,
    force_rebuild_data=True,
    force_rerun_benchmark=True,
    benchmark_overrides=None,
)
scenario_overrides = SCENARIO_PRESETS[SCENARIO_NAME]


## Run and validate

In [ ]:
state = run_notebook_workflow(
    run_config,
    scenario_name=SCENARIO_NAME,
    scenario_overrides=scenario_overrides,
)

# Familiar aliases; `state` remains the authoritative carrier.
COUNTRY = state.country_code
prepared = state.prepared
data = prepared.data
cfg = prepared.cfg
simulation = state.simulation
model = state.model
df_base = state.df_base
benchmark = state.benchmark
df_benchmark = state.df_benchmark

validate_notebook_state(state)


## Macro, fiscal, policy, labour, and balance sheets

In [ ]:
plot_output(
    df=df_base[list(MACRO_COLUMNS)],
    no_rows=5,
    no_cols=4,
    country_code=COUNTRY,
    line_color="#1f77b4",
    **FIGURE_SIZES["benchmark"],
)
plot_output(df=df_base[list(FISCAL_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_base[list(POLICY_COLUMNS)], no_rows=4, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_base[list(LABOUR_COLUMNS)], no_rows=2, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_base[list(BALANCE_SHEET_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])


## Benchmark comparison

In [ ]:
if df_benchmark is None:
    print("Benchmark disabled in Inputs.")
else:
    plot_output(df=df_base[list(MACRO_COLUMNS)], df_compare=df_benchmark[list(MACRO_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    plot_output(df=df_base[list(FISCAL_COLUMNS)], df_compare=df_benchmark[list(FISCAL_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    plot_output(df=df_base[list(POLICY_COLUMNS)], df_compare=df_benchmark[list(POLICY_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    plot_output(df=df_base[list(BALANCE_SHEET_COLUMNS)], df_compare=df_benchmark[list(BALANCE_SHEET_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])


## Permanent-income decomposition

In [ ]:
decomposition = build_permanent_income_log_ratio_decomposition_df(
    simulation,
    country_code=COUNTRY,
    reducer="mean",
    include_log_real_pc_income=True,
)
plot_permanent_income_log_ratio_decomposition(
    simulation,
    country_code=COUNTRY,
    columns=["ln_y_p_over_y", "common_log_ratio"],
    reducer="mean",
)
contributions = build_permanent_income_forecast_contribution_table(
    simulation,
    country_code=COUNTRY,
    periods=list(range(9)),
    include_fixed=False,
)
contributions


## Sector and firm-credit diagnostics

In [ ]:
firm_sector_groups_table(model, COUNTRY)
plot_cumulative_insolvent_firms_by_sector(df_base)

credit_panels = [
    ["total_target_short_term_credit", "total_received_short_term_credit"],
    ["total_target_long_term_credit", "total_received_long_term_credit"],
    "short_term_loan_debt",
    "long_term_loan_debt",
]
plot_agent_timeseries(
    model,
    COUNTRY,
    "firms",
    variables=credit_panels,
    agg="sum",
    no_cols=2,
    show_legend=False,
    **FIGURE_SIZES["dense"],
)
plot_firm_credit_to_equity_and_capital(model, COUNTRY, show=True, return_df=False)
plot_sector_tfp_investment_desired_mb_mc_ratio(model, COUNTRY)
